# Práctica 1

Manuel Suano Gallardo

## Ejercicio 1

### E 1.1.

En la carpeta donde se encuentre este notebook debe estar los tres CSV que se van a utilizar en esta práctica.

Comenzaremos subiendo el archivo de datos `countries.csv` y `clientes.csv`

In [2]:
!hdfs dfs -mkdir /practica1
!hdfs dfs -mkdir /practica1/ejercicio1
!hdfs dfs -put /media/notebooks/Practica1/countries.csv /practica1/ejercicio1
!hdfs dfs -put /media/notebooks/Practica1/clients.csv /practica1/ejercicio1
!hdfs dfs -ls /practica1/ejercicio1

Found 2 items
-rw-r--r--   3 root supergroup       1289 2025-11-15 21:03 /practica1/ejercicio1/clients.csv
-rw-r--r--   3 root supergroup       4120 2025-11-15 21:03 /practica1/ejercicio1/countries.csv


Primero comprobramos la primera fila de cada archivo, por si pudieran llevar el nombre de las columnas

In [3]:
with open('./clients.csv', "r") as fp:
    print("Primera línea clients.csv: " + fp.readline())

with open('./countries.csv', "r") as fp:
    print("Primera línea countries.csv: " + fp.readline())

Primera línea clients.csv: Bertram Pearcy  ,bueno,SO

Primera línea countries.csv: Name,Code



Se puede observar que el archivo clients.csv no tiene el nombre de las columnas, mientras que countries.csv sí. Pasamos a ver un poco el contenido.

In [4]:
import pandas as pd

doc_countries = pd.read_csv('./countries.csv', header=0)

doc_countries.head()

,Name,Code
0,Afghanistan,AF
1,Åland Islands,AX
2,Albania,AL
3,Algeria,DZ
4,American Samoa,AS


In [5]:
doc_clients = pd.read_csv('./clients.csv', names=['Cliente','Opinion','Pais'])

doc_clients.head()

,Cliente,Opinion,Pais
0,Bertram Pearcy,bueno,SO
1,Steven Ulman,regular,ZA
2,Enid Follansbee,malo,GS
3,Candie Jacko,malo,SS
4,Alana Zufelt,regular,ES


A partir de esta información podemos constuir el mapper. Este recibirá los fichero y mostrará por la stdout, tanto los códigos de los países de cada valoración "bueno", como los pares de equivalencia entre el código del país y el nombre de este. Hay que tener en cuenta que la cabecera del archivo `countries.csv` también se recibirá pero no hay que enviarla.

```
mapper:
    equiv_cod_paises <- {}
    paises_bueno <- []

    para cada linea de stdin hacer
        div_linea <- dividir(linea, ',')

        si longitud(div_linea) = 3 y div_linea[1] es "bueno" entonces
            imprimir(div_linea[2])
        si longitud(div_linea) = 2 y div_linea[0] no es "Name" entonces
            imprimir(div_linea[0], div_linea[1])
```


In [6]:
%%writefile 1_1_mapper.py
#!/usr/bin/env python3

import sys

equiv_cod_paises = {}
paises_bueno = []

for linea in sys.stdin:
    # Dividimos la línea
    div_linea = linea.strip().split(',')

    # Es del fichero de opiniones y tiene por opinión "bueno"
    if len(div_linea) == 3 and div_linea[1] == 'bueno':
        # Se añade el país
        print(div_linea[2])
        
    # Es del fichero de países y no es la primera línea (el nombre de las columnas)
    elif len(div_linea) == 2 and div_linea[0] != 'Name':
        # Se manda la equivalencia entre ficheros
        print(f"{div_linea[0]},{div_linea[1]}")

Writing 1_1_mapper.py


El reducer irá realizando, por un lado, el conteo de los códigos, mientras que por otro se almacenarán las equivalencias para que cuando se haya terminado de recibir la información se pueda sustituir el código del país por el nombre, que quedará asociado al conteo realizado. Para mostrarlo por la stdout se ordenará los países alfabéticamente.

```
    reducer:
        conteo_valoraciones_paises <- {}
        equivalencia_paises <- {}

        para cada linea de stdin hacer
            si linea contiene ',' entonces
                nombre_pais, codigo_pais <- dividir(linea, ',')
                equivalencia_paises[codigo_pais] <- nombre_pais
            sino entonces
                # la línea es un código de país
                si linea no está en conteo_valoraciones_paises entonces
                    conteo_valoraciones_paises[linea] <- 0
                conteo_valoraciones_paises[linea] <- conteo_valoraciones_paises[linea] + 1

        conteo_valoraciones_paises <- renombrar_claves(conteo_valoraciones_paises, equivalencia_paises)

        para cada pais en ordenar(conteo_valoraciones_paises) hacer
            imprimir(pais, conteo_valoraciones_paises[pais])
```

In [48]:
%%writefile 1_1_reducer.py
#!/usr/bin/env python3

import sys

# Conteo de países
conteo_valoraciones_paises = {}
equivalencia_paises = {}

for linea in sys.stdin:
    linea = linea.strip()

    # Se trata de una equivalencia entre países
    if ',' in linea:
        linea = linea.split(',')
        nombre_pais = linea[0]
        cod_pais = linea[1]

        equivalencia_paises[cod_pais] = nombre_pais
    
    # Se trata de un código de un país con una valoración "bueno"
    else:
        if linea not in conteo_valoraciones_paises:
            conteo_valoraciones_paises[linea] = 0
        conteo_valoraciones_paises[linea] += 1

# Sustituimos las claves por el nombre de los paises
conteo_valoraciones_paises = { 
    equivalencia_paises[cod_pais] : conteo for cod_pais, conteo in conteo_valoraciones_paises.items() 
}

# Los mostramos ordenados alfabéticamente
for pais in sorted(conteo_valoraciones_paises):
    print(f'["{pais}"]\t{conteo_valoraciones_paises[pais]}')


Overwriting 1_1_reducer.py


Una vez se tiene el mapper y el reducer, realizamos la ejecución de Hadoop Streaming.

In [ ]:
!hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
-files 1_1_mapper.py,1_1_reducer.py \
-mapper 1_1_mapper.py \
-reducer 1_1_reducer.py \
-input /practica1/ejercicio1/clients.csv \
-input /practica1/ejercicio1/countries.csv \
-output /practica1/ejercicio1/salida_1_1

Visualizamos la salida del MapReduce

In [9]:
!hdfs dfs -cat /practica1/ejercicio1/salida_1_1/part-00000

["Canada"]	1
["Guam"]	3
["Guinea"]	1
["Portugal"]	1
["Qatar"]	1
["Somalia"]	1
["South Africa"]	1
["South Georgia and the South Sandwich Islands"]	1
["South Sudan"]	1
["Spain"]	3
["Turkey"]	1
["United States"]	1


### E 1.2.

Para obtener el país con mayor número de valoraciones "bueno" simplemente tendremos que recorrer el diccionario buscando el máximo. Al finalizar el bucle lo mostraremos por la stdout junto con el conteo asociado. El mapper se mantendrá el mismo.

```
    reducer:
        conteo_valoraciones_paises <- {}
        equivalencia_paises <- {}

        para cada linea de stdin hacer
            si linea contiene ',' entonces
                nombre_pais, codigo_pais <- dividir(linea, ',')
                equivalencia_paises[codigo_pais] <- nombre_pais
            sino entonces
                # la línea es un código de país
                si linea no está en conteo_valoraciones_paises entonces
                    conteo_valoraciones_paises[linea] <- 0
                conteo_valoraciones_paises[linea] <- conteo_valoraciones_paises[linea] + 1

        conteo_valoraciones_paises <- renombrar_claves(conteo_valoraciones_paises, equivalencia_paises)

        maximo <- -1
        pais_maximo <- Null
        para cada pais en conteo_valoraciones_paises hacer
            si conteo_valoraciones_paises[pais] > maximo entonces
                maximo <- conteo_valoraciones_paises[pais]
                pais_maximo <- pais

        imprimir(maximo, pais_maximo)
```

In [60]:
%%writefile 1_2_reducer.py
#!/usr/bin/env python3

import sys

# Conteo de países
conteo_valoraciones_paises = {}
equivalencia_paises = {}

for linea in sys.stdin:
    linea = linea.strip()

    # Se trata de una equivalencia entre países
    if ',' in linea:
        linea = linea.split(',')
        nombre_pais = linea[0]
        cod_pais = linea[1]

        equivalencia_paises[cod_pais] = nombre_pais
    
    # Se trata de un código de un país con una valoración "bueno"
    else:
        if linea not in conteo_valoraciones_paises:
            conteo_valoraciones_paises[linea] = 0
        conteo_valoraciones_paises[linea] += 1

# Sustituimos las claves por el nombre de los paises
conteo_valoraciones_paises = { 
    equivalencia_paises[cod_pais] : conteo for cod_pais, conteo in conteo_valoraciones_paises.items() 
}

# Recorremos buscando el país con mayor número de valoraciones
maximo = -1
pais_maximo = None
# Quitamos el orden alfabético por país
for pais in conteo_valoraciones_paises:
    if conteo_valoraciones_paises[pais] > maximo:
        maximo = conteo_valoraciones_paises[pais]
        pais_maximo = pais
    
print(f'{maximo}\t["{pais_maximo}"]')

Overwriting 1_2_reducer.py


In [62]:
!hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
-files 1_1_mapper.py,1_2_reducer.py \
-mapper 1_1_mapper.py \
-reducer 1_2_reducer.py \
-input /practica1/ejercicio1/clients.csv \
-input /practica1/ejercicio1/countries.csv \
-output /practica1/ejercicio1/salida_1_2

packageJobJar: [/tmp/hadoop-unjar1688439390115541363/] [] /tmp/streamjob8927471139713643079.jar tmpDir=null
2025-11-15 22:13:36,053 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2025-11-15 22:13:36,194 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2025-11-15 22:13:36,379 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1763222702967_0020
2025-11-15 22:13:36,735 INFO mapred.FileInputFormat: Total input files to process : 2
2025-11-15 22:13:36,840 INFO mapreduce.JobSubmitter: number of splits:3
2025-11-15 22:13:36,968 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1763222702967_0020
2025-11-15 22:13:36,968 INFO mapreduce.JobSubmitter: Executing with tokens: []
2025-11-15 22:13:37,162 INFO conf.Configuration: resource-types.xml not found
2025-11-15 22:13:37,162 INFO resource.ResourceUtils

Visualizamos la salida del MapReduce

In [63]:
!hdfs dfs -cat /practica1/ejercicio1/salida_1_2/part-00000

3	["Spain"]


### E 1.3.

Para este caso, en el bucle de mapper donde se busca el máximo, en lugar de guardar únicamente aquel que supera al máximo, en caso de encontrar otro país que igual lo que vamos a hacer es almacenarlo. Si se encuentra alguno que supere al máximo entonces tendremos que borrar la lista y empezar una nueva con el nuevo valor. De nuevo el mapper será el mismo.

```
    reducer:
        conteo_valoraciones_paises <- {}
        equivalencia_paises <- {}

        para cada linea de stdin hacer
            si linea contiene ',' entonces
                nombre_pais, codigo_pais <- dividir(linea, ',')
                equivalencia_paises[codigo_pais] <- nombre_pais
            sino entonces
                # la línea es un código de país
                si linea no está en conteo_valoraciones_paises entonces
                    conteo_valoraciones_paises[linea] <- 0
                conteo_valoraciones_paises[linea] <- conteo_valoraciones_paises[linea] + 1

        conteo_valoraciones_paises <- renombrar_claves(conteo_valoraciones_paises, equivalencia_paises)

        maximo <- -1
        pais_maximo <- []
        para cada pais en conteo_valoraciones_paises hacer
            si conteo_valoraciones_paises[pais] > maximo entonces
                maximo <- conteo_valoraciones_paises[pais]
                pais_maximo <- [pais]
            si conteo_valoraciones_paises[pais] = maximo entonces
                pais_maximo <- añadir(pais_maximo, pais)

        para cada pais en ordenado(pais_maximo) hacer
            imprimir(maximo, pais)
```

In [74]:
%%writefile 1_3_reducer.py
#!/usr/bin/env python3

import sys

# Conteo de países
conteo_valoraciones_paises = {}
equivalencia_paises = {}

for linea in sys.stdin:
    linea = linea.strip()

    # Se trata de una equivalencia entre países
    if ',' in linea:
        linea = linea.split(',')
        nombre_pais = linea[0]
        cod_pais = linea[1]

        equivalencia_paises[cod_pais] = nombre_pais
    
    # Se trata de un código de un país con una valoración "bueno"
    else:
        if linea not in conteo_valoraciones_paises:
            conteo_valoraciones_paises[linea] = 0
        conteo_valoraciones_paises[linea] += 1

# Sustituimos las claves por el nombre de los paises
conteo_valoraciones_paises = { 
    equivalencia_paises[cod_pais] : conteo for cod_pais, conteo in conteo_valoraciones_paises.items() 
}

# Recorremos buscando el país con mayor número de valoraciones
maximo = -1
paises_maximo = []
# Quitamos el orden alfabético por país
for pais in conteo_valoraciones_paises:
    if conteo_valoraciones_paises[pais] > maximo:
        maximo = conteo_valoraciones_paises[pais]
        paises_maximo = [pais]
    # Si es el mismo valor, se almacena
    elif conteo_valoraciones_paises[pais] == maximo:
        paises_maximo.append(pais)

# Ahora simplemente recorremos los países ordenados alfabéticamente
for pais_maximo in sorted(paises_maximo):
    print(f'{maximo}\t["{pais_maximo}"]')

Overwriting 1_3_reducer.py


In [78]:
!hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
-files 1_1_mapper.py,1_3_reducer.py \
-mapper 1_1_mapper.py \
-reducer 1_3_reducer.py \
-input /practica1/ejercicio1/clients.csv \
-input /practica1/ejercicio1/countries.csv \
-output /practica1/ejercicio1/salida_1_3

packageJobJar: [/tmp/hadoop-unjar3142157789730139195/] [] /tmp/streamjob7733212348678289346.jar tmpDir=null
2025-11-15 22:26:58,924 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2025-11-15 22:26:59,092 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2025-11-15 22:26:59,303 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1763222702967_0023
2025-11-15 22:26:59,757 INFO mapred.FileInputFormat: Total input files to process : 2
2025-11-15 22:26:59,881 INFO mapreduce.JobSubmitter: number of splits:3
2025-11-15 22:27:00,038 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1763222702967_0023
2025-11-15 22:27:00,039 INFO mapreduce.JobSubmitter: Executing with tokens: []
2025-11-15 22:27:00,246 INFO conf.Configuration: resource-types.xml not found
2025-11-15 22:27:00,247 INFO resource.ResourceUtils

In [79]:
!hdfs dfs -cat /practica1/ejercicio1/salida_1_3/part-00000

3	["Guam"]
3	["Spain"]


## Ejercicio 2

Antes de nada, vamos a crear el directorio en Hadoop y vamos a subir el CSV

In [368]:
!hdfs dfs -mkdir /practica1/ejercicio2
!hdfs dfs -put /media/notebooks/Practica1/convocatorias-2020.csv /practica1/ejercicio2
!hdfs dfs -ls /practica1/ejercicio2

Found 1 items
-rw-r--r--   3 root supergroup    5675992 2025-11-16 19:32 /practica1/ejercicio2/convocatorias-2020.csv


Una vez subido el archivo, empezamos con el script en HiveQL para la creación de la tabla que almacenará estos datos. A pesar de estar decrito en la práctica de qué trata cada campo, voy a leer la primera línea para comprobar el formato del CSV y posteriormente leeré las 5 primeras líneas del fichero para comprobar como podría encontrarme cada campo.

In [369]:
with open('./convocatorias-2020.csv', 'r') as dataset:
    print(dataset.readline())

40/51-0159-LPR19,381,381 - Estado Mayor General de La Fuerza Aérea,40,Departamento Contrataciones Córdoba - UOC 40/51,Licitacion Privada,Sin Modalidad,,2019,02/01/2020 08:00:00 a.m.,10/01/2020 10:00:00 a.m.,Única,Nacional,“Adquisición de Elementos de Librería para el HOSPITAL AERONÁUTICO CÓRDOBA,“Adquisición de Elementos de Librería para el HOSPITAL AERONÁUTICO CÓRDOBA,1957142.84,Proceso de Compra



Podemos ver que la separación del archivo es por comas. Procedemos a leer las 5 primeras líneas.

In [ ]:
primeras_lineas = []
with open('./convocatorias-2020.csv') as dataset:
    for linea in range(0,5):
        # Añadimos la línea sin el último carácter (\n) y dividida por el ,
        primeras_lineas.append(dataset.readline()[:-1].split(','))

pd.DataFrame(data=primeras_lineas, columns=range(0,len(primeras_lineas[0])))

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,40/51-0159-LPR19,381,381 - Estado Mayor General de La Fuerza Aérea,40,Departamento Contrataciones Córdoba - UOC 40/51,Licitacion Privada,Sin Modalidad,,2019,02/01/2020 08:00:00 a.m.,10/01/2020 10:00:00 a.m.,Única,Nacional,“Adquisición de Elementos de Librería para el ...,“Adquisición de Elementos de Librería para el ...,1957142.84,Proceso de Compra
1,22-0041-CDI19,250,250 - Caja de Retiros Jubilaciones y Pensiones...,22,22 - Dpto de Compras y Suministros - Caja de R...,Contratación Directa,Sin Modalidad,Apartado 1: Compulsa Abreviada Por Monto,2019,02/01/2020 12:00:00 p.m.,13/01/2020 09:00:00 a.m.,Única,Nacional,Servicio de seguridad de información sensible ...,Servicio de seguridad de información sensible ...,119915.25,Proceso de Compra
2,40/51-0918-CDI19,381,381 - Estado Mayor General de La Fuerza Aérea,40,Departamento Contrataciones Córdoba - UOC 40/51,Contratación Directa,Sin Modalidad,Apartado 1: Compulsa Abreviada Por Monto,2019,02/01/2020 12:00:00 p.m.,13/01/2020 10:00:00 a.m.,Única,Nacional,“Servicio Médico Asistencial para Organismos V...,“Servicio Médico Asistencial para Organismos V...,99000,Proceso de Compra
3,92-0075-CDI19,116,116 - Biblioteca Nacional,92,92 - Departamento de Compras y Contrataciones- BN,Contratación Directa,Sin Modalidad,Apartado 1: Compulsa Abreviada Por Monto,2019,02/01/2020 03:00:00 p.m.,27/01/2020 11:00:00 a.m.,Única,Nacional,ADQUISICIÓN DE CAJAS Y REVISTEROS PARA LA DIRE...,ADQUISICIÓN DE CAJAS Y REVISTEROS PARA LA DIRE...,1450050,Proceso de Compra
4,46/1-0005-CDI20,604,604 - Dirección Nacional de Vialidad,46,46/01 - 19° Bahía Blanca - DNV,Contratación Directa,Sin Modalidad,Apartado 1: Compulsa Abreviada Por Monto,2020,02/01/2020 02:00:00 p.m.,08/01/2020 10:00:00 a.m.,Única,Nacional,SERVICIO DE ARMADO DE REFUGIO PEATONAL (SEGUND...,SERVICIO DE ARMADO DE REFUGIO PEATONAL,684000,Proceso de Compra


A partir de esta información pasamos a definir la tabla en un script:

In [371]:
%%writefile crear-tabla-convocatoria_intermedia.hql

DROP TABLE IF EXISTS convocatoria_intermedia;
CREATE TABLE convocatoria_intermedia(
    n_procedimiento STRING,
    n_saf INT,
    desc_saf STRING,
    n_uoc INT,
    desc_uoc STRING,
    tipo_procedimiento STRING,
    modalidad STRING,
    apartado_directa STRING,
    ejercicio INT,
    fecha_publicacion STRING,
    fecha_apertura STRING,
    etapa STRING,
    alcance STRING,
    nom_procedimiento STRING,
    obj_procedimiento STRING,
    monto_estimado DOUBLE,
    tipo_operacion STRING
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
WITH SERDEPROPERTIES (
    "separatorChar" = ",",
    "quoteChar"     = "\"",
    "escapeChar"    = "\\"
)
STORED AS TEXTFILE
TBLPROPERTIES (
    "serialization.encoding" = "UTF-8"
);


Overwriting crear-tabla-convocatoria_intermedia.hql


In [372]:
%%writefile crear-tabla-convocatoria.hql

DROP TABLE IF EXISTS convocatoria;
CREATE TABLE convocatoria(
    n_procedimiento STRING,
    n_saf INT,
    desc_saf STRING,
    n_uoc INT,
    desc_uoc STRING,
    tipo_procedimiento STRING,
    modalidad STRING,
    apartado_directa STRING,
    ejercicio INT,
    fecha_publicacion TIMESTAMP,
    fecha_apertura TIMESTAMP,
    etapa STRING,
    alcance STRING,
    nom_procedimiento STRING,
    obj_procedimiento STRING,
    monto_estimado DOUBLE,
    tipo_operacion STRING
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
WITH SERDEPROPERTIES (
    "separatorChar" = ",",
    "quoteChar"     = "\"",
    "escapeChar"    = "\\"
)
STORED AS TEXTFILE
TBLPROPERTIES (
    "serialization.encoding" = "UTF-8"
);

Overwriting crear-tabla-convocatoria.hql


Las tablas son internas por dos razones:
1. Al ser datos del 2020 no van a ser modificados o se van a añadir nuevos. Si fuera para datos del año actual, donde es posible que se añadan nuevos datos, entonces sí que tendría sentido que se crease externa, para que dichas modificaciones, hechas por procesos externos, fueran automáticas.
2. Va a ser una tabla temporal donde se almacenen los datos para posteriormente pasarlas a la tabla final, donde las fechas están adaptadas al tipo que le corresponde.

Además, tampoco se cumple ninguna de las recomendaciones indicadas en [esta guia](https://learn.microsoft.com/es-es/archive/blogs/cindygross/hdinsight-hive-internal-and-external-tables-intro) de Microsoft.

La decisión de crear una segunda tabla se debe a que en los campos donde va una fecha se encuentra los indicadores a.m. y p.m., que no son aceptados como tal, sino que deben ser am o AM y pm o PM. Esta primera tabla servirá para indroducir los datos y la segunda tabla para transformarlos. Se hace la segunda tabla por la comodidad de tener las fechas guardadas como tipo `TIMESTAMP` y no como `STRING`.

Proseguimos con la ejecución de ambos scripts usando beeline.

In [373]:
!beeline -u "jdbc:hive2://" -f crear-tabla-convocatoria_intermedia.hql

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

In [374]:
!beeline -u "jdbc:hive2://" -f crear-tabla-convocatoria.hql

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

Procedemos a comprobar si se han creado

In [375]:
!beeline -u "jdbc:hive2://" -e "show tables;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

En vista de que se han creado pasamos a añadir los datos a las tablas. Primero la añadiremos a `convocatoria_intermedia` y, posteriormente, tras preprocesar las fechas, añadiremos los datos a convocatoria.

In [376]:
!beeline -u "jdbc:hive2://" -e "LOAD DATA INPATH '/practica1/ejercicio2/convocatorias-2020.csv' INTO TABLE convocatoria_intermedia;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

Mostramos los 5 primeros campos, pero solo algunas columnas para una mejor visualización.

In [378]:
!beeline -u "jdbc:hive2://" -e "SELECT n_procedimiento,n_saf,fecha_apertura,tipo_procedimiento FROM convocatoria_intermedia LIMIT 5;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

Ahora pasamos los datos a la tabla definitiva, donde estarán almacenados las fechas correctamente. Para ello hacemos la transformación de las fechas: primero eliminamos los . en a.m. y p.m., después hacemos la transformación de la fecha obtenida a timestamp, pasando la fecha a un timestamp de unix y posteriormente a un formato de fecha que es reconocido por Hive (https://cwiki.apache.org/confluence/pages/viewpage.action?pageId=27362046#LanguageManualUDF-DateFunctions).

In [379]:
%%writefile insert_into_convocatoria.hql

INSERT INTO TABLE convocatoria
SELECT     
    n_procedimiento,n_saf,desc_saf,n_uoc,desc_uoc,tipo_procedimiento,modalidad,apartado_directa,ejercicio,
    from_unixtime(
        unix_timestamp(
            regexp_replace(fecha_publicacion, '\\.', ''),
            'dd/MM/yyyy hh:mm:ss a'
        )
    ) AS fecha_publicacion,
    from_unixtime(
        unix_timestamp(
            regexp_replace(fecha_apertura, '\\.', ''),
            'dd/MM/yyyy hh:mm:ss a'
        )
    ) AS fecha_apertura,
    etapa,alcance,nom_procedimiento,obj_procedimiento,monto_estimado,tipo_operacion
FROM convocatoria_intermedia;

Writing insert_into_convocatoria.hql


In [380]:
!beeline -u "jdbc:hive2://" -f insert_into_convocatoria.hql

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

Podemos ver como se han producido las transformaciones de las fechas a formato 24H:

In [381]:
!beeline -u "jdbc:hive2://" -e "SELECT n_procedimiento,fecha_publicacion,fecha_apertura FROM convocatoria LIMIT 5;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

A partir de esta tabla podemos realizar las consultas

1. Devuelve los registros cuyo tipo de procedimiento de contratación sea contratación directa.

No consigo transformar hacer que las tildes sean leídas como tal, por lo que no puedo hacer la consulta:
```SELECT n_procedimiento,tipo_procedimiento FROM convocatoria WHERE tipo_procedimiento = 'Contratación Directa' LIMIT 5;```
que es la que me gustaría hacer.

In [383]:
!beeline -u "jdbc:hive2://" -e "SELECT n_procedimiento,tipo_procedimiento FROM convocatoria WHERE tipo_procedimiento LIKE '%Directa%' LIMIT 5;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

2. Devuelve los registros cuyo tipo de procedimiento de contratación sea contratación directa y cuyo ejercicio fiscal sea anterior a 2020.

In [385]:
!beeline -u "jdbc:hive2://" -e "SELECT n_procedimiento,tipo_procedimiento,ejercicio FROM convocatoria WHERE tipo_procedimiento LIKE '%Directa%' AND ejercicio < 2020 LIMIT 5;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

3. ¿Cuántos tipos de SAF tenemos para los registros de las convocatorias? Hay que contar el número de valores únicos de la columna que contiene las descripciones del Servicio Administrativo Financiero (Descripcion SAF).

In [389]:
!beeline -u "jdbc:hive2://" -e "SELECT COUNT(DISTINCT(desc_saf)) FROM convocatoria;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

4. ¿Cuál es el máximo presupuesto estimado de estas convocatorias? Utilizaremos el campo monto estimado de la tabla convocatorias.

In [408]:
!beeline -u "jdbc:hive2://" -e "SELECT MAX(CAST(monto_estimado AS DECIMAL(20,0))) FROM convocatoria;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

5. ¿Cuál es el monto estimado medio que se emplea en las convocatorias agrupado por su alcance (nacional e internacional)? 

In [409]:
!beeline -u "jdbc:hive2://" -e "SELECT AVG(CAST(monto_estimado AS DECIMAL(20,0))) FROM convocatoria;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

### Ejercicio 2.4

Para este ejercico se ha escogido el registro de los accidentes de tráficos ocurridos en la ciudad de Madrid durante el año 2024. Se puede obtener desde la siguiente [página](https://datos.gob.es/es/catalogo/l01280796-accidentes-de-trafico-de-la-ciudad-de-madrid1).

In [410]:
! wget https://datos.madrid.es/egob/catalogo/300228-28-accidentes-trafico-detalle.csv -O /media/notebooks/Practica1/2024_accidentalidad.csv 

--2025-11-17 20:11:12--  https://datos.madrid.es/egob/catalogo/300228-28-accidentes-trafico-detalle.csv
Resolving datos.madrid.es (datos.madrid.es)... 2.18.188.31, 2.18.188.10
Connecting to datos.madrid.es (datos.madrid.es)|2.18.188.31|:443... connected.
HTTP request sent, awaiting response... 302 Moved Temporarily
Location: https://datos.madrid.es:443/egobfiles/MANUAL/300228/2024_Accidentalidad.csv [following]
--2025-11-17 20:11:13--  https://datos.madrid.es/egobfiles/MANUAL/300228/2024_Accidentalidad.csv
Reusing existing connection to datos.madrid.es:443.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/csv]
Saving to: ‘/media/notebooks/Practica1/2024_accidentalidad.csv’

/media/notebooks/Pr     [     <=>            ]   9.07M  10.8MB/s    in 0.8s    

2025-11-17 20:11:14 (10.8 MB/s) - ‘/media/notebooks/Practica1/2024_accidentalidad.csv’ saved [9513218]



Vamos a abrir el archivo para comprobar de qué datos está compuesto. El archivo cuenta en su primera línea con el nombre de las columnas.

In [424]:
data = pd.read_csv('./2024_accidentalidad.csv', header=0, sep=';')

data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49340 entries, 0 to 49339
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   num_expediente        49340 non-null  object 
 1   fecha                 49340 non-null  object 
 2   hora                  49340 non-null  object 
 3   localizacion          49340 non-null  object 
 4   numero                49340 non-null  object 
 5   cod_distrito          49340 non-null  int64  
 6   distrito              49340 non-null  object 
 7   tipo_accidente        49335 non-null  object 
 8   estado_meteorológico  43424 non-null  object 
 9   tipo_vehiculo         48946 non-null  object 
 10  tipo_persona          49340 non-null  object 
 11  rango_edad            49340 non-null  object 
 12  sexo                  49340 non-null  object 
 13  cod_lesividad         27344 non-null  float64
 14  lesividad             27344 non-null  object 
 15  coordenada_x_utm   

Y vamos a ver las primeras líneas

In [419]:
data.head()

,num_expediente,fecha,hora,localizacion,numero,cod_distrito,distrito,tipo_accidente,estado_meteorológico,tipo_vehiculo,tipo_persona,rango_edad,sexo,cod_lesividad,lesividad,coordenada_x_utm,coordenada_y_utm,positiva_alcohol,positiva_droga
0,2023S040280,04/01/2024,14:09:00,AVDA. NICETO ALCALA ZAMORA / AUTOV. M-11,3,16,HORTALEZA,Colisión fronto-lateral,Lluvia débil,Motocicleta > 125cc,Conductor,De 55 a 59 años,Hombre,2.0,Ingreso inferior o igual a 24 horas,444913.0,4481427.0,N,NaN
1,2023S040280,04/01/2024,14:09:00,AVDA. NICETO ALCALA ZAMORA / AUTOV. M-11,3,16,HORTALEZA,Colisión fronto-lateral,Lluvia débil,Turismo,Conductor,De 55 a 59 años,Mujer,14.0,Sin asistencia sanitaria,444913.0,4481427.0,N,NaN
2,2023S040309,15/02/2024,14:05:00,CALL. TESORO / CALL. MINAS,18,1,CENTRO,Colisión fronto-lateral,Lluvia débil,Bicicleta,Conductor,De 25 a 29 años,Hombre,7.0,Asistencia sanitaria sólo en el lugar del acci...,440123.0,4475170.0,N,NaN
3,2023S040309,15/02/2024,14:05:00,CALL. TESORO / CALL. MINAS,18,1,CENTRO,Colisión fronto-lateral,Lluvia débil,Motocicleta hasta 125cc,Conductor,De 35 a 39 años,Hombre,14.0,Sin asistencia sanitaria,440123.0,4475170.0,N,NaN
4,2023S040310,18/02/2024,10:40:00,GTA. RUIZ JIMENEZ / CALL. SAN BERNARDO,3,7,CHAMBERÍ,Colisión lateral,Despejado,Turismo,Conductor,De 25 a 29 años,Hombre,NaN,NaN,440137.0,4475721.0,N,NaN


Hay algunas que no cuadra muy bien el tipo inferido con lo que se presupone leyendo el nombre de la columna, concretamente el caso de la columna `numero` y de `positiva_droga`. Vamos a comprobar que valores se pueden encontrar ahí para que se infieran esos tipos.

Por un lado, para la columna `positiva_droga` parace ser que hay muchos valores NaN, por lo que partiremos descartando esos valores. Podemos ver que los valores que contiene para indicar positivo en droga es el `1.0`.

In [426]:
data[data['positiva_droga'].notnull()]['positiva_droga']

195      1.0
217      1.0
237      1.0
489      1.0
589      1.0
        ... 
48148    1.0
48379    1.0
48601    1.0
48718    1.0
49107    1.0
Name: positiva_droga, Length: 203, dtype: float64

Por otro lado, para la columna `numero` vamos a utilizar una expresión regular para comprobar aquellos campos que no sean únicamente números, para comprobar que aparece. La expresión regular será `^(?!\d+).*$`. Podemos ver como hay valores que combianan números con letras y el carácter `+`.

In [427]:
data[data['numero'].str.match('^(?!\d+).*$')]['numero']

168      +00500E
169      +00500E
257      +00500S
321      +00600E
343      +00600S
          ...   
48878    +00500S
48879    +00500S
48903    +00600E
49034    +00500E
49110    +00700E
Name: numero, Length: 1072, dtype: object

Una vez hemos visto los datos vamos a subir el archivo al HDFS:

In [446]:
!hdfs dfs -put /media/notebooks/Practica1/2024_accidentalidad.csv /practica1/ejercicio2
!hdfs dfs -ls /practica1/ejercicio2

Found 1 items
-rw-r--r--   3 root supergroup    9513218 2025-11-17 21:10 /practica1/ejercicio2/2024_accidentalidad.csv


Y ahora vamos a crear la tabla usando las información que hemos obtenido arriba.

In [444]:
%%writefile crear_tabla_accidentes.hql

DROP TABLE IF EXISTS accidentes;
CREATE TABLE accidentes(
    num_expediente STRING,
    fecha DATE,
    hora TIMESTAMP,
    localizacion STRING,
    numero STRING,
    cod_distrito INT,
    distrito STRING,
    tipo_accidente STRING,
    estado_meteo STRING,
    tipo_vehiculo STRING,
    tipo_persona STRING,
    rango_edad STRING,
    sexo STRING,
    cod_lesividad DOUBLE,
    lesividad STRING,
    x_utm DOUBLE,
    y_utm DOUBLE,
    positiva_alcohol STRING,
    positiva_droga DOUBLE
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
WITH SERDEPROPERTIES (
    "separatorChar" = ";",
    "quoteChar"     = "\"",
    "escapeChar"    = "\\"
)
STORED AS TEXTFILE
TBLPROPERTIES (
    "serialization.encoding" = "UTF-8",
    "skip.header.line.count"="1"
);

Overwriting crear_tabla_accidentes.hql


Creamos la tabla dentro de Hive

In [445]:
!beeline -u "jdbc:hive2://" -f crear_tabla_accidentes.hql

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

Subimos el fichero a la tabla

In [447]:
!beeline -u "jdbc:hive2://" -e "LOAD DATA INPATH '/practica1/ejercicio2/2024_accidentalidad.csv' INTO TABLE accidentes;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

Y comprobamos que se hayan cargado los datos

In [448]:
!beeline -u "jdbc:hive2://" -e "SELECT num_expediente,fecha,hora,localizacion FROM accidentes LIMIT 5;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

Primera consulta: Número de accidentes, por sexo, donde la edad estaba entre 25 y 29 años, habían dado positivo en alcohol y se encontraba lloviendo

In [449]:
%%writefile consulta_1.hql

SELECT sexo, COUNT(*) as total_por_sexo
FROM accidentes
WHERE rango_edad LIKE '%25 a 29%' AND positiva_alcohol = 'S' AND estado_meteo LIKE '%Lluvia%'
GROUP BY sexo;

Overwriting consulta_1.hql


In [450]:
!beeline -u "jdbc:hive2://" -f consulta_1.hql

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

Segunda consulta: Número de accidentes ocurridos en una avenida por alcance en personas de 30 a 50 años, agrupados por tipo de vehículo

In [452]:
%%writefile consulta_2.hql

SELECT tipo_vehiculo, COUNT(*) as total_por_vehiculo
FROM accidentes
WHERE (rango_edad LIKE '%30 a 34%' OR rango_edad LIKE '%35 a 39%' OR rango_edad LIKE '%40 a 44%' OR rango_edad LIKE '%45 a 50%') AND tipo_accidente = 'Alcance' AND localizacion LIKE '%AVDA.%'
GROUP BY tipo_vehiculo;

Writing consulta_2.hql


In [455]:
!beeline -u "jdbc:hive2://" -f consulta_2.hql

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session